# Sales & Demand Forecasting with Linear Regression
### Course template: Machine Learning for Business Insights (Module 3)

**What this notebook does**
1. Loads your CSV and checks data quality
2. Builds a leakage-safe preprocessing pipeline
3. Compares a "predict the average" baseline against Linear Regression, Ridge, and Lasso
4. Tunes regularization with cross-validation
5. Evaluates the model with business metrics (MAE, RMSE, R²) and puts error in plain terms
6. Extracts coefficients and explains what drives your sales

**How to use it**
- Edit **only the CONFIG cell** (Cell 3) on your first run
- Run every cell top to bottom (Runtime → Run all in Colab)
- Read the printed notes after each step, since they tell you what to look for

**Leave `CSV_PATH = None`** to run on a built-in demo dataset first.

In [ ]:
# ============================================================
# IMPORTS
# These are the libraries the notebook uses. All of them come
# preinstalled in Google Colab. In local Jupyter, install with:
#   pip install pandas numpy scikit-learn matplotlib seaborn
# ============================================================

import warnings
warnings.filterwarnings("ignore")  # keeps output readable for beginners

import numpy as np                  # math on arrays of numbers
import pandas as pd                 # tables of data (like Excel in Python)
import matplotlib.pyplot as plt     # charts
import seaborn as sns               # nicer-looking charts

# scikit-learn: the main machine learning library
from sklearn.model_selection import (
    train_test_split,     # split data into training and test sets
    KFold,                # standard cross-validation
    RepeatedKFold,        # cross-validation repeated several times (small data)
    TimeSeriesSplit,      # cross-validation that respects time order
    cross_validate,       # score a model across many folds
    GridSearchCV,         # try every combination of settings
    RandomizedSearchCV,   # try a random sample of settings
    learning_curve,       # does more data help?
)
from sklearn.pipeline import Pipeline                  # chains prep + model
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer               # fills blanks
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor               # "predict the average" baseline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone                         # copies a model's settings for refitting
from scipy.stats import loguniform                     # for log-scale random search

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_STATE = 42  # fixes randomness so results repeat exactly each run
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: EDIT THIS CELL FOR YOUR DATA
# ============================================================

# --- 1. Where is your data? ---------------------------------
# Put your CSV file path here, e.g. "/content/my_sales.csv" in Colab
# (upload the file using the folder icon on the left first).
# Leave as None to use the built-in demo dataset.
CSV_PATH = None

# --- 2. What are you predicting? ----------------------------
# The column with the number you want to forecast:
# sales, revenue, units sold, demand, etc.
TARGET_COLUMN = "weekly_sales"

# --- 3. Which columns are inputs? ---------------------------
# NUMERIC: real numbers (price, ad spend, temperature, staff count)
NUMERIC_COLUMNS = ["price", "ad_spend", "store_size_sqft", "staff_count", "competitor_price"]

# CATEGORICAL: labels with no natural order (region, store type, channel)
CATEGORICAL_COLUMNS = ["region", "promo_type"]

# Columns to exclude completely: IDs, free text, and anything you
# would NOT know on the day you make the forecast (LEAKAGE!)
# Example of leakage: "returns_this_week" when forecasting this week's sales.
DROP_COLUMNS = ["store_id"]

# --- 4. Is your data time-ordered? --------------------------
# If each row is a week/month and you forecast the future, set the
# date column name here. The notebook will then sort by date and
# validate by training on the past and testing on the future.
# Leave as None if rows are not ordered in time.
DATE_COLUMN = "week_start"

# --- 5. Target transform -------------------------------------
# Sales and revenue are often skewed (many small values, a few huge ones).
# True = model log(sales) instead of sales. Coefficients then read as
# approximate % effects. Start with False; switch if residual plots
# fan out (see Section 8).
LOG_TARGET = False

# --- 6. Data size mode ---------------------------------------
# "auto" picks based on row count. You can force "small" or "large".
DATA_SIZE_MODE = "auto"

# --- 7. Tuning method ----------------------------------------
# "grid" = try every value in a small list (best for small data)
# "random" = try a random sample from wide ranges (good for larger data)
TUNING_METHOD = "grid"

# --- 8. Large-data option ------------------------------------
# If your file is huge (e.g., millions of rows), explore on a sample
# first for speed. None = use all rows.
SAMPLE_ROWS_FOR_EXPLORATION = None   # e.g., 100_000

# --- 9. Business context (used in plain-English summaries) ---
UNIT_LABEL = "$"   # "$", "units", "orders", etc.
TEST_SIZE = 0.2    # share of data held back for the final test

print("Config set. Target:", TARGET_COLUMN)

In [ ]:
# ============================================================
# LOAD DATA
# If CSV_PATH is None, we generate a realistic fake dataset so you
# can see the whole notebook work before using your own file.
# The demo data is SYNTHETIC: do not draw real conclusions from it.
# ============================================================

def make_demo_data(n_stores=12, n_weeks=20, seed=RANDOM_STATE):
    """Creates a small synthetic weekly sales dataset (240 rows)."""
    rng = np.random.default_rng(seed)
    rows = []
    regions = ["North", "South", "West"]
    promos = ["none", "discount", "bundle"]
    start = pd.Timestamp("2025-01-06")
    for s in range(n_stores):
        region = regions[s % 3]
        size = rng.integers(2_000, 8_000)
        for w in range(n_weeks):
            price = rng.normal(20, 2)
            ad = max(0, rng.normal(1_500, 500))
            staff = rng.integers(3, 12)
            comp = price + rng.normal(0, 1.5)
            promo = rng.choice(promos, p=[0.6, 0.25, 0.15])
            sales = (
                5_000
                - 350 * price
                + 2.2 * ad
                + 1.1 * size
                + 400 * staff
                + 250 * comp
                + {"none": 0, "discount": 2_500, "bundle": 1_200}[promo]
                + {"North": 0, "South": -1_500, "West": 1_000}[region]
                + rng.normal(0, 2_500)
            )
            rows.append({
                "store_id": f"S{s:02d}",
                "week_start": start + pd.Timedelta(weeks=w),
                "region": region,
                "promo_type": promo,
                "price": round(price, 2),
                "ad_spend": round(ad, 0),
                "store_size_sqft": size,
                "staff_count": staff,
                "competitor_price": round(comp, 2),
                "weekly_sales": round(max(sales, 0), 0),
            })
    df = pd.DataFrame(rows)
    # Add a few blanks so the data-quality checks have something to find
    df.loc[rng.choice(len(df), 8, replace=False), "ad_spend"] = np.nan
    df.loc[rng.choice(len(df), 5, replace=False), "promo_type"] = np.nan
    return df

if CSV_PATH is None:
    df = make_demo_data()
    print("Using DEMO data (synthetic). Set CSV_PATH in the CONFIG cell to use your own.")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")

# Parse the date column if there is one
if DATE_COLUMN and DATE_COLUMN in df.columns:
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")

# Optional sampling for very large files
if SAMPLE_ROWS_FOR_EXPLORATION and len(df) > SAMPLE_ROWS_FOR_EXPLORATION:
    df = df.sample(SAMPLE_ROWS_FOR_EXPLORATION, random_state=RANDOM_STATE)
    print(f"Sampled {SAMPLE_ROWS_FOR_EXPLORATION:,} rows for exploration. "
          "Re-run with SAMPLE_ROWS_FOR_EXPLORATION = None for final results.")

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK
# Catches typos in column names before anything else breaks.
# ============================================================

all_config_cols = [TARGET_COLUMN] + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
missing = [c for c in all_config_cols if c not in df.columns]
if missing:
    raise ValueError(f"These columns are in CONFIG but not in your data: {missing}\n"
                     f"Your columns are: {list(df.columns)}")

# Make sure numeric columns really are numbers (e.g., "$1,200" stored as text)
for col in NUMERIC_COLUMNS + [TARGET_COLUMN]:
    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(r"[$,%\s]", "", regex=True),
            errors="coerce",
        )
        print(f"Converted '{col}' from text to numbers. Check for new blanks below.")

# Rows with no target can't be used for training
before = len(df)
df = df.dropna(subset=[TARGET_COLUMN])
if len(df) < before:
    print(f"Dropped {before - len(df)} rows with a blank target.")

unused = [c for c in df.columns
          if c not in all_config_cols + DROP_COLUMNS + ([DATE_COLUMN] if DATE_COLUMN else [])]
if unused:
    print(f"Note: these columns are NOT being used: {unused}")
print("Config check passed.")

## Section 1: Quick EDA and Data Quality

Spend five minutes here. Most modeling problems are really data problems.
Look for: blanks, impossible values (negative sales, $0 prices), duplicates,
and a target that looks very skewed.

In [ ]:
# ============================================================
# DATA QUALITY REPORT
# ============================================================

print("=== Column types ===")
print(df.dtypes, "\n")

print("=== Missing values (count and %) ===")
miss = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct": (df.isna().mean() * 100).round(1),
})
print(miss[miss["missing"] > 0] if miss["missing"].sum() else "No missing values.", "\n")

print("=== Duplicate rows ===")
print(df.duplicated().sum(), "exact duplicates\n")

print("=== Numeric summary ===")
display(df[NUMERIC_COLUMNS + [TARGET_COLUMN]].describe().T)

print("=== Category counts (watch for typos and rare values) ===")
for col in CATEGORICAL_COLUMNS:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

# Simple red flags
if (df[TARGET_COLUMN] < 0).any():
    print(f"\nWARNING: {TARGET_COLUMN} has negative values. Are these returns or errors?")

In [ ]:
# ============================================================
# VISUAL CHECKS
# Left: how is the target distributed? A long right tail suggests
#       LOG_TARGET = True may help.
# Right: correlation of numeric inputs with the target and each other.
#        Inputs correlated above ~0.8 with EACH OTHER make coefficients
#        unstable (multicollinearity, Module 3, Lesson 3.4).
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.histplot(df[TARGET_COLUMN], bins=30, kde=True, ax=axes[0])
axes[0].set_title(f"Distribution of {TARGET_COLUMN}")
skew = df[TARGET_COLUMN].skew()
axes[0].text(0.98, 0.95, f"skew = {skew:.2f}", transform=axes[0].transAxes, ha="right", va="top")

corr = df[NUMERIC_COLUMNS + [TARGET_COLUMN]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Correlations")
plt.tight_layout()
plt.show()

if skew > 1:
    print("Target is right-skewed (skew > 1). Consider LOG_TARGET = True.")

# Flag highly correlated input pairs
high_pairs = [
    (a, b, corr.loc[a, b])
    for i, a in enumerate(NUMERIC_COLUMNS)
    for b in NUMERIC_COLUMNS[i + 1:]
    if abs(corr.loc[a, b]) > 0.8
]
for a, b, r in high_pairs:
    print(f"High overlap: {a} & {b} (r = {r:.2f}). Consider dropping one or combining them.")

# If time-ordered, show the trend
if DATE_COLUMN and DATE_COLUMN in df.columns:
    trend = df.groupby(DATE_COLUMN)[TARGET_COLUMN].sum()
    trend.plot(figsize=(14, 3.5), title=f"Total {TARGET_COLUMN} over time")
    plt.show()

In [ ]:
# ============================================================
# DATA SIZE CHECK
# Decides how cautious the rest of the notebook should be.
# Rule of thumb: aim for 10–20 rows per input feature.
# One-hot encoding turns each category value into its own feature,
# so we count those too.
# ============================================================

n_rows = len(df)
n_features_est = len(NUMERIC_COLUMNS) + sum(df[c].nunique() - 1 for c in CATEGORICAL_COLUMNS)
rows_per_feature = n_rows / max(n_features_est, 1)

if DATA_SIZE_MODE == "auto":
    MODE = "small" if n_rows < 1_000 else "large"
else:
    MODE = DATA_SIZE_MODE

print(f"Rows: {n_rows:,}  |  Estimated features after encoding: {n_features_est}")
print(f"Rows per feature: {rows_per_feature:.1f}")
print(f"Mode: {MODE.upper()}")

if rows_per_feature < 10:
    print("\nWARNING: fewer than 10 rows per feature. High overfitting risk.\n"
          "  → Drop weaker inputs, group rare categories, and rely on Ridge.")
if MODE == "small":
    print("\nSMALL-DATA PATH: repeated cross-validation, small tuning grids,\n"
          "stronger regularization, and a stability check on coefficients.")
else:
    print("\nLARGE-DATA PATH: standard 5-fold CV and wider tuning ranges.")

## Section 2: Train/Test Split and Validation Strategy

We hold back a **test set** the model never sees until the very end. It's our final, honest check.

- **Time-ordered data:** the test set is the **most recent** periods (you can't train on the future).
- **Not time-ordered:** a random 20% is held back.

**Cross-validation** (used for comparing and tuning models) rotates through the training data
several times, so one lucky or unlucky split doesn't decide anything.

In [ ]:
# ============================================================
# SPLIT INTO TRAIN AND TEST
# ============================================================

feature_cols = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
time_ordered = bool(DATE_COLUMN) and DATE_COLUMN in df.columns

if time_ordered:
    # Sort by date: oldest first. The last TEST_SIZE share is the test set.
    df = df.sort_values(DATE_COLUMN).reset_index(drop=True)
    cutoff = int(len(df) * (1 - TEST_SIZE))
    train_df, test_df = df.iloc[:cutoff], df.iloc[cutoff:]
    print(f"Time-based split: train up to {train_df[DATE_COLUMN].max().date()}, "
          f"test from {test_df[DATE_COLUMN].min().date()}")
else:
    train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    print("Random split.")

X_train, y_train = train_df[feature_cols], train_df[TARGET_COLUMN]
X_test, y_test = test_df[feature_cols], test_df[TARGET_COLUMN]
print(f"Train rows: {len(X_train):,}  |  Test rows: {len(X_test):,}")

# ============================================================
# CROSS-VALIDATION STRATEGY
# ============================================================
if time_ordered:
    # Each fold trains on earlier rows and validates on the next chunk.
    # Note: with several stores per week, a fold boundary can split a week;
    # that's acceptable for a first pass.
    CV = TimeSeriesSplit(n_splits=5 if MODE == "large" else 4)
    print("CV: TimeSeriesSplit (train on past, validate on future)")
elif MODE == "small":
    # 5 folds, repeated 5 times with different shuffles = 25 scores.
    # Steadier estimate when every row counts.
    # (For under ~50 rows, LeaveOneOut is another option.)
    CV = RepeatedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    print("CV: RepeatedKFold (5 folds × 5 repeats)")
else:
    CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    print("CV: KFold (5 folds)")

## Section 3: Leakage-Safe Preprocessing Pipeline

The pipeline bundles every prep step **with** the model. During cross-validation,
fill values and scaling are learned **only from the training folds**, so the
validation data never "leaks" into training.

| Column type | Steps |
|---|---|
| Numeric | Fill blanks with the median → scale to a common range |
| Categorical | Fill blanks with "Unknown" → one-hot encode (one yes/no column per value) |

Scaling matters here: Ridge and Lasso penalize coefficient size, so all inputs need to be on comparable scales.

In [ ]:
# ============================================================
# PREPROCESSOR
# ============================================================

numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),  # blanks → middle value
    ("scale", StandardScaler()),                   # put on a common scale
])

# REFERENCE GROUPS: the most common value of each category (from TRAINING rows).
# One-hot encoding drops this value, and every other value's coefficient reads
# as "compared to the most common group", which is the easiest comparison to explain.
REFERENCE_GROUPS = [X_train[c].mode().iloc[0] for c in CATEGORICAL_COLUMNS]
print("Reference groups:", {c: str(r) for c, r in zip(CATEGORICAL_COLUMNS, REFERENCE_GROUPS)})

categorical_steps = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
    # drop=REFERENCE_GROUPS removes the reference value from each column.
    # handle_unknown="ignore" means a new category in future data won't crash the model
    # (it is treated like the reference group).
    ("onehot", OneHotEncoder(drop=REFERENCE_GROUPS, handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_steps, NUMERIC_COLUMNS),
    ("cat", categorical_steps, CATEGORICAL_COLUMNS),
])

def make_model(regressor):
    """Wraps prep + model into one pipeline, with optional log target."""
    pipe = Pipeline([("prep", preprocessor), ("model", regressor)])
    if LOG_TARGET:
        # Model learns log(1 + sales); predictions are converted back automatically.
        return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)
    return pipe

print("Preprocessor ready. (If you see an error about 'sparse_output', "
      "upgrade scikit-learn: pip install -U scikit-learn)")

## Section 4: Baseline and Default Models

Every model must beat the **baseline**: always predicting the training average.
If it can't, the inputs aren't telling us anything useful yet.

We compare three models with default settings:
- **Linear Regression:** plain, no guardrails
- **Ridge:** shrinks all coefficients a bit (the safest default, especially on small data)
- **Lasso:** can shrink weak coefficients to exactly zero (automatic feature selection)

In [ ]:
# ============================================================
# MODEL COMPARISON (cross-validated on TRAINING data only)
# Each model is trained and scored once per fold. We report the
# average score and the spread (±) across folds.
# return_train_score=True also shows the TRAINING score, so we can
# see overfitting: a big train-vs-validation gap means the model
# memorized the training rows instead of learning the pattern.
# ============================================================

SCORING = {
    "MAE": "neg_mean_absolute_error",       # typical miss, in target units
    "RMSE": "neg_root_mean_squared_error",  # like MAE but punishes big misses more
    "R2": "r2",                             # share of variation explained (1.0 = perfect)
}

candidates = {
    "Baseline (predict the average)": make_model(DummyRegressor(strategy="mean")),
    "Linear Regression": make_model(LinearRegression()),
    "Ridge (alpha=1)": make_model(Ridge(alpha=1.0)),
    "Lasso (alpha=1)": make_model(Lasso(alpha=1.0, max_iter=20_000)),
}

def cv_summary(model, name):
    """Cross-validates one model and returns a tidy row of results."""
    res = cross_validate(model, X_train, y_train, cv=CV, scoring=SCORING,
                         return_train_score=True, error_score=np.nan)
    row = {"model": name}
    for k, s in SCORING.items():
        sign = -1 if s.startswith("neg_") else 1      # sklearn stores errors as negatives
        row[f"{k}"] = sign * np.nanmean(res[f"test_{k}"])
        row[f"{k} ±"] = np.nanstd(res[f"test_{k}"])
        row[f"{k} (train)"] = sign * np.nanmean(res[f"train_{k}"])
    return row

results = pd.DataFrame([cv_summary(m, n) for n, m in candidates.items()]).set_index("model")
display(results[["MAE", "MAE ±", "MAE (train)", "RMSE", "RMSE ±", "R2", "R2 ±", "R2 (train)"]])

base_mae = results.loc["Baseline (predict the average)", "MAE"]
best_name = results["MAE"].idxmin()
print(f"""
HOW TO READ THIS
- MAE: the typical miss, in {UNIT_LABEL}. Lower is better. The baseline misses by about {UNIT_LABEL}{base_mae:,.0f}.
- RMSE: like MAE, but big misses count extra. If RMSE is much bigger than MAE, a few large misses dominate.
- R2: share of the ups and downs in {TARGET_COLUMN} the model explains. 0 = no better than the average.
- ±: spread across folds. Differences smaller than the spread are effectively ties.
- (train) vs. validation: a large gap means overfitting. Linear models usually show a small gap.

Best model by validation MAE: {best_name}
It improves on the baseline by {UNIT_LABEL}{base_mae - results.loc[best_name, 'MAE']:,.0f} per prediction
({(1 - results.loc[best_name, 'MAE'] / base_mae):.0%} smaller typical miss).
""")
if results.loc[best_name, "MAE"] >= base_mae * 0.97:
    print("⚠ No model clearly beats the baseline. Revisit your inputs before tuning: "
          "are they really known ahead of time, and do they plausibly drive the target?")

## Section 5: Tune the Regularization Strength

Ridge and Lasso have one main setting, **alpha**: how hard the model is pulled toward "no effect".

| alpha | What happens | When it's right |
|---|---|---|
| Very small (0.001) | Almost plain Linear Regression | Lots of data, few inputs |
| Medium (1–10) | Coefficients shrink a little; more stable | Most business datasets |
| Large (100+) | Coefficients shrink a lot; very cautious | Small data, overlapping inputs |

We try a range of alpha values with cross-validation, then apply the **simplicity rule**:
among all settings within **one standard error** of the best score (the fold spread ± divided by √folds),
pick the **largest alpha** (most cautious model).
A slightly more cautious model that scores the same is the safer bet on new data.

In [ ]:
# ============================================================
# TUNING ALPHA FOR RIDGE AND LASSO
# TUNING_METHOD "grid" tries a fixed list; "random" samples from a
# range on a log scale (0.001, 0.01, 0.1, ... are equally likely).
# ============================================================

TUNE_METRIC = "neg_mean_absolute_error"     # we tune on MAE: easiest to explain
# When LOG_TARGET=True, the pipeline is wrapped, so the setting name gets a prefix
ALPHA_PARAM = "regressor__model__alpha" if LOG_TARGET else "model__alpha"

if MODE == "small":
    ridge_grid = [0.1, 0.3, 1, 3, 10, 30, 100]          # leans cautious
    lasso_grid = [0.1, 0.3, 1, 3, 10, 30, 100]
else:
    ridge_grid = list(np.logspace(-3, 3, 13))           # 0.001 → 1000
    lasso_grid = list(np.logspace(-3, 3, 13))
if LOG_TARGET:
    # On the log scale, effects are small numbers, so Lasso needs tiny alphas
    lasso_grid = [a / 1_000 for a in lasso_grid]

def tune_alpha(regressor, grid, name):
    """Searches alpha, prints a results table, and applies the simplicity rule."""
    pipe = make_model(regressor)
    if TUNING_METHOD == "random":
        search = RandomizedSearchCV(pipe, {ALPHA_PARAM: loguniform(min(grid), max(grid))},
                                    n_iter=20, cv=CV, scoring=TUNE_METRIC,
                                    random_state=RANDOM_STATE, n_jobs=-1)
    else:
        search = GridSearchCV(pipe, {ALPHA_PARAM: grid}, cv=CV, scoring=TUNE_METRIC, n_jobs=-1)
    search.fit(X_train, y_train)

    tbl = pd.DataFrame({
        "alpha": [p[ALPHA_PARAM] for p in search.cv_results_["params"]],
        "MAE": -search.cv_results_["mean_test_score"],
        "±": search.cv_results_["std_test_score"],
    }).sort_values("alpha").reset_index(drop=True)

    best = tbl.loc[tbl["MAE"].idxmin()]
    # SIMPLICITY RULE: biggest alpha whose MAE is within ONE STANDARD ERROR of the best
    # (standard error = fold spread ÷ √number of folds)
    se = best["±"] / np.sqrt(CV.get_n_splits())
    within = tbl[tbl["MAE"] <= best["MAE"] + se]
    chosen = within.loc[within["alpha"].idxmax()]
    print(f"{name}: best alpha {best['alpha']:.4g} (MAE {best['MAE']:,.1f} ± {best['±']:,.1f}) → "
          f"CHOSEN alpha {chosen['alpha']:.4g} (MAE {chosen['MAE']:,.1f})")
    return tbl, float(chosen["alpha"]), float(chosen["MAE"]), float(chosen["±"])

ridge_tbl, ridge_alpha, ridge_mae, ridge_sd = tune_alpha(Ridge(), ridge_grid, "Ridge")
lasso_tbl, lasso_alpha, lasso_mae, lasso_sd = tune_alpha(Lasso(max_iter=20_000), lasso_grid, "Lasso")

# Chart: validation MAE vs. alpha (log scale). Look for a flat valley.
fig, ax = plt.subplots(figsize=(8, 4))
for tbl, lab in [(ridge_tbl, "Ridge"), (lasso_tbl, "Lasso")]:
    ax.errorbar(tbl["alpha"], tbl["MAE"], yerr=tbl["±"], fmt="o-", capsize=3, label=lab)
ax.set_xscale("log"); ax.set_xlabel("alpha (log scale)"); ax.set_ylabel(f"Validation MAE ({UNIT_LABEL})")
ax.set_title("Regularization strength vs. typical miss"); ax.legend(); plt.show()

# Pick the final model. Ridge is the default (keeps every driver and is
# steadier on small data); Lasso wins only if it's clearly better.
if lasso_mae < ridge_mae - ridge_sd:
    final_model, final_name = make_model(Lasso(alpha=lasso_alpha, max_iter=20_000)), f"Lasso (alpha={lasso_alpha:.3g})"
else:
    final_model, final_name = make_model(Ridge(alpha=ridge_alpha)), f"Ridge (alpha={ridge_alpha:.3g})"
print(f"\nFINAL MODEL: {final_name}")
print("(Ridge is preferred unless Lasso beats it by more than one spread.)")

## Section 6: Final Test-Set Check

Now, and only now, we score the **test set** the model has never seen. This is the number to quote
to stakeholders. Run this step once. If you go back and tweak things based on test results, the test
set stops being an honest check.

In [ ]:
# ============================================================
# FINAL EVALUATION ON THE TEST SET
# ============================================================

final_model.fit(X_train, y_train)
pred_test = final_model.predict(X_test)

baseline = make_model(DummyRegressor(strategy="mean")).fit(X_train, y_train)
pred_base = baseline.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)
rmse = np.sqrt(mean_squared_error(y_test, pred_test))
r2 = r2_score(y_test, pred_test)
mae_base = mean_absolute_error(y_test, pred_base)
avg_actual = y_test.mean()

summary = pd.DataFrame({
    "Baseline (average)": [mae_base, np.sqrt(mean_squared_error(y_test, pred_base)), r2_score(y_test, pred_base)],
    final_name: [mae, rmse, r2],
}, index=["MAE", "RMSE", "R2"])
display(summary)

print(f"""
IN PLAIN TERMS
- On average, the forecast is off by about {UNIT_LABEL}{mae:,.0f} per row,
  which is {mae / avg_actual:.0%} of the average {TARGET_COLUMN} ({UNIT_LABEL}{avg_actual:,.0f}).
- Guessing the average would be off by {UNIT_LABEL}{mae_base:,.0f}, so the model cuts the typical miss by {(1 - mae / mae_base):.0%}.
- The model explains {r2:.0%} of the ups and downs in {TARGET_COLUMN}.
- Compare the test MAE with the cross-validated MAE from Section 5. If test is MUCH worse,
  something changed between training and test periods (or the CV setup leaked).
""")

## Section 7: What Drives the Target? (Coefficients in Business Units)

Each coefficient answers: **"If this input goes up by one unit and everything else stays the same,
how much does the prediction change?"**

- **Numeric inputs** are shown two ways: per **1 unit** (e.g. +$1 of price) and per **typical change**
  (one standard deviation, the size of a normal swing in your data). The second is better for ranking drivers.
- **Categories** are compared with the **reference group** (the one dropped during encoding).
- With `LOG_TARGET = True`, effects read as **approximate % changes** instead of units.

⚠️ Coefficients describe **associations in your data**, not guaranteed cause and effect.
Treat them as strong leads to test, not proof.

In [ ]:
# ============================================================
# COEFFICIENTS → BUSINESS UNITS
# ============================================================

# Unwrap the fitted pipeline (TransformedTargetRegressor hides it one level down)
fitted_pipe = final_model.regressor_ if LOG_TARGET else final_model
prep = fitted_pipe.named_steps["prep"]
coefs = fitted_pipe.named_steps["model"].coef_
names = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]

scaler = prep.named_transformers_["num"].named_steps["scale"]
n_num = len(NUMERIC_COLUMNS)

rows = []
for i, (name, c) in enumerate(zip(names, coefs)):
    if i < n_num:
        sd = scaler.scale_[i]
        per_unit, per_typical = c / sd, c            # model saw scaled inputs: 1 scaled unit = 1 SD
        kind, step = "numeric", f"+1 (typical swing ±{sd:,.2f})"
    else:
        per_unit = per_typical = c                   # yes/no column: effect of being in this group
        kind, step = "category vs. reference", "in this group"
    rows.append({"driver": name, "type": kind, "step": step,
                 "effect per 1 unit": per_unit, "effect per typical change": per_typical})
coef_df = pd.DataFrame(rows)

if LOG_TARGET:
    # On the log scale, exp(coef) - 1 = % change in the target
    for col in ["effect per 1 unit", "effect per typical change"]:
        coef_df[col] = (np.exp(coef_df[col]) - 1) * 100
    unit_txt = "%"
else:
    unit_txt = UNIT_LABEL

coef_df["abs"] = coef_df["effect per typical change"].abs()
coef_df = coef_df.sort_values("abs", ascending=False).drop(columns="abs").reset_index(drop=True)
display(coef_df)

# Reference groups (what each category is compared against)
ohe = prep.named_transformers_["cat"].named_steps["onehot"]
for col, cats, d in zip(CATEGORICAL_COLUMNS, ohe.categories_, ohe.drop_idx_):
    print(f"Reference group for {col}: '{cats[d]}' (category effects are compared with this group)")

# Chart: ranked drivers by typical-change effect
plot_df = coef_df.sort_values("effect per typical change")
colors = ["indianred" if v < 0 else "seagreen" for v in plot_df["effect per typical change"]]
plt.figure(figsize=(8, 0.4 * len(plot_df) + 1.5))
plt.barh(plot_df["driver"], plot_df["effect per typical change"], color=colors)
plt.axvline(0, color="black", lw=0.8)
plt.title(f"Drivers of {TARGET_COLUMN} (effect of a typical change, {unit_txt})")
plt.xlabel(f"Change in predicted {TARGET_COLUMN} ({unit_txt})"); plt.tight_layout(); plt.show()

# Plain-English sentences for the top drivers
print("PLAIN-ENGLISH SUMMARY (holding everything else constant):")
for _, r in coef_df.head(6).iterrows():
    direction = "raises" if r["effect per typical change"] > 0 else "lowers"
    if r["type"] == "numeric":
        amt = abs(r["effect per 1 unit"])
        txt = f"{amt:,.2f}%" if LOG_TARGET else f"{UNIT_LABEL}{amt:,.2f}"
        print(f"- Each +1 in {r['driver']} {direction} predicted {TARGET_COLUMN} by about {txt}.")
    else:
        amt = abs(r["effect per 1 unit"])
        txt = f"{amt:,.1f}%" if LOG_TARGET else f"{UNIT_LABEL}{amt:,.0f}"
        print(f"- Being '{r['driver']}' {direction} predicted {TARGET_COLUMN} by about {txt} vs. the reference group.")
if final_name.startswith("Lasso"):
    dropped = coef_df.loc[coef_df["effect per 1 unit"].abs() < 1e-9, "driver"].tolist()
    print(f"\nLasso set these drivers to zero (no reliable effect): {dropped or 'none'}")

In [ ]:
# ============================================================
# COEFFICIENT STABILITY CHECK (most useful on SMALL data)
# We refit the model on many resampled versions of the training data.
# If a driver's sign flips between + and −, don't build a decision on it.
# ============================================================

N_BOOT = 100 if MODE == "small" else 30
rng = np.random.default_rng(RANDOM_STATE)
boot = []
for b in range(N_BOOT):
    idx = rng.integers(0, len(X_train), len(X_train))          # sample rows with replacement
    m = clone(final_model).fit(X_train.iloc[idx], y_train.iloc[idx])
    pipe_b = m.regressor_ if LOG_TARGET else m
    boot.append(pipe_b.named_steps["model"].coef_)
boot = pd.DataFrame(boot, columns=names)

stab = pd.DataFrame({
    "median (scaled units)": boot.median(),
    "low (5%)": boot.quantile(0.05),
    "high (95%)": boot.quantile(0.95),
    "% same sign as median": (np.sign(boot) == np.sign(boot.median())).mean() * 100,
}).sort_values("% same sign as median")
display(stab)
shaky = stab.index[stab["% same sign as median"] < 90].tolist()
print("Drivers whose direction is NOT reliable (sign agrees < 90% of the time):", shaky or "none")
print("Tip: report the reliable drivers with confidence; describe shaky ones as 'unclear'.")

## Section 8: Residual Checks (Where Does the Model Miss?)

A **residual** is actual minus predicted. Good residuals look like random noise around zero.

| What you see | What it means | What to try |
|---|---|---|
| Residuals fan out as predictions grow | Bigger stores/weeks have bigger misses | `LOG_TARGET = True` |
| A curve (U-shape) | A relationship isn't a straight line | Add a squared term, or try a tree model (Module 5) |
| A few huge misses | Outliers or special events | Check those rows; add an event flag |
| Residuals drift over time | The business changed | Retrain more often; add a trend input |

In [ ]:
# ============================================================
# RESIDUAL PLOTS (test set)
# ============================================================
resid = y_test - pred_test

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(y_test, pred_test, alpha=0.6)
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
axes[0].plot(lims, lims, "r--", label="perfect forecast")
axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted"); axes[0].set_title("Actual vs. predicted"); axes[0].legend()

axes[1].scatter(pred_test, resid, alpha=0.6)
axes[1].axhline(0, color="red", ls="--")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual (actual − predicted)")
axes[1].set_title("Residuals vs. predicted (look for fans or curves)")

sns.histplot(resid, kde=True, ax=axes[2])
axes[2].set_title("Distribution of misses")
plt.tight_layout(); plt.show()

# Simple automatic hints
lo, hi = np.percentile(pred_test, [33, 67])
spread_low = resid[pred_test <= lo].abs().mean()
spread_high = resid[pred_test >= hi].abs().mean()
if spread_high > 1.5 * spread_low and not LOG_TARGET:
    print("Misses grow with the size of the prediction → try LOG_TARGET = True in CONFIG.")
big = resid.abs() > 3 * resid.std()
print(f"Rows with very large misses (>3 SD): {int(big.sum())}. Inspect them:")
display(test_df.loc[big[big].index].head(10) if big.any() else pd.DataFrame())

if time_ordered:
    plt.figure(figsize=(12, 3.5))
    plt.plot(test_df[DATE_COLUMN].values, resid.values, "o", alpha=0.5)
    plt.axhline(0, color="red", ls="--"); plt.title("Residuals over time (should hover around zero)")
    plt.show()

In [ ]:
# ============================================================
# LEARNING CURVE: would more data help?
# If the validation line is still rising at the right edge, collecting
# more history is likely to improve accuracy. If both lines have
# flattened and are close together, better INPUTS matter more than more rows.
# ============================================================
lc_cv = CV if not isinstance(CV, RepeatedKFold) else KFold(5, shuffle=True, random_state=RANDOM_STATE)
sizes, tr_sc, va_sc = learning_curve(final_model, X_train, y_train, cv=lc_cv,
                                     scoring="neg_mean_absolute_error",
                                     train_sizes=np.linspace(0.3, 1.0, 6), n_jobs=-1)
plt.figure(figsize=(7, 4))
plt.plot(sizes, -tr_sc.mean(1), "o-", label="Training MAE")
plt.plot(sizes, -va_sc.mean(1), "o-", label="Validation MAE")
plt.xlabel("Training rows"); plt.ylabel(f"MAE ({UNIT_LABEL})"); plt.title("Learning curve"); plt.legend(); plt.show()

## Section 9: Small Data vs. Large Data: What Changes

**Small data (under ~1,000 rows, or under 10–20 rows per input):**
- Use **Ridge**, and let the simplicity rule pick a cautious alpha
- Keep inputs to the ones with a clear business reason; group rare categories into "Other"
- Use repeated cross-validation (done automatically) and trust the **stability check** above
- Report ranges ("about $300–$400 per $1 of price"), not single precise numbers

**Large data (tens of thousands of rows and up):**
- Plain Linear Regression and Ridge usually agree; regularization matters less
- `TUNING_METHOD = "random"` with wide ranges is fast and fine
- Explore on a sample (`SAMPLE_ROWS_FOR_EXPLORATION`), then rerun on everything
- Small effects become "statistically real" but may not matter. Ask whether the size of the effect is worth acting on
- If a straight line underfits (curved residuals), try the tree models in Module 5

In [ ]:
# ============================================================
# SAVE THE MODEL (and a scoring example)
# The saved file contains the WHOLE pipeline: prep + model.
# Load it later and call .predict() on new rows with the same columns.
# ============================================================
import joblib

final_model.fit(pd.concat([X_train, X_test]), pd.concat([y_train, y_test]))   # refit on all data for use
joblib.dump(final_model, "linear_regression_model.joblib")
print("Saved: linear_regression_model.joblib (refit on train + test)")

# --- How to use it next month --------------------------------
# model = joblib.load("linear_regression_model.joblib")
# new_rows = pd.read_csv("next_week_plan.csv")       # same input columns as training
# new_rows["forecast"] = model.predict(new_rows[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS])

# Optional: save test-set predictions for review in Excel
out = test_df.copy()
out["predicted"] = pred_test.round(1)
out["miss"] = (out[TARGET_COLUMN] - out["predicted"]).round(1)
out.to_csv("linear_regression_test_predictions.csv", index=False)
print("Saved: linear_regression_test_predictions.csv")

## Final Checklist

- [ ] CONFIG matches my data (target, inputs, date column, leakage columns dropped)
- [ ] Every input is known **before** the moment I forecast
- [ ] The model clearly beats the "predict the average" baseline
- [ ] I tuned alpha with cross-validation and used the simplicity rule (one standard error)
- [ ] I looked at the test set **once**, at the end
- [ ] I can state the typical miss in business terms ("about $X per week, Y% of average")
- [ ] Top drivers make business sense, and their direction is stable in the bootstrap check
- [ ] Residual plots show no strong fan or curve (or I switched on `LOG_TARGET`)
- [ ] I described drivers as associations to test, not proven causes
- [ ] Model saved, with a note of the date and data range it was trained on

**Next step:** if residuals curve or accuracy plateaus, try the Decision Tree and Gradient Boosting
notebooks on the same data and compare their test MAE with this one.